# Train Expert 802 on Kaggle's free GPU

This notebook contains the repository's native training source code. It does **not** clone GitHub at runtime, so it can start in Kaggle sessions where Internet is disabled. It trains the native Expert 802 Transformer from random initialization; no hosted generation API or pretrained model is used.

## Before running
1. In Kaggle Notebook Settings, select **GPU** under Accelerator. The notebook cannot train while the log says `Accelerator: None`.
2. Add a Kaggle Dataset containing your reviewed, legally usable corpus. Its root must contain `corpus/` and `corpus_manifest.csv`. The CSV columns are `path,license,source_url,permission_basis,notes`; `path` is relative to `corpus/`.
3. Set `CORPUS_DATASET` below to the mounted dataset folder name.
4. The Kaggle image must already include PyTorch, NumPy and the `tokenizers` package. The notebook checks this and reports any missing package; it does not attempt a network install.

Each session exports `/kaggle/working/expert802-session.zip`. Download it, upload it as a **private Kaggle Dataset**, attach it to the next session as `expert802-previous-session`, and increase `TOTAL_STEPS` to resume. GPU availability and session duration depend on Kaggle quotas.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, zipfile

# CHANGE THIS to the exact mounted folder name under /kaggle/input.
CORPUS_DATASET = "expert802-licensed-corpus"
PREVIOUS_SESSION_DATASET = "expert802-previous-session"

# TOTAL cumulative steps. For example, first run 3000, next session 6000.
TOTAL_STEPS = 3000
BATCH_SIZE = 4
CONTEXT_LENGTH = 1024
VOCAB_SIZE = 8000
START_FRESH = False

CORPUS_DIR = Path("/kaggle/input") / CORPUS_DATASET / "corpus"
MANIFEST = Path("/kaggle/input") / CORPUS_DATASET / "corpus_manifest.csv"
PREVIOUS_DIR = Path("/kaggle/input") / PREVIOUS_SESSION_DATASET
WORKSPACE = Path("/kaggle/working/expert_802_AI")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
print("Workspace:", WORKSPACE)
print("Corpus exists:", CORPUS_DIR.exists(), "Manifest exists:", MANIFEST.exists())


In [ ]:
# Write the exact reviewed training source into this Kaggle session; no GitHub clone is needed.
SOURCE_FILES = {
  "model.py": "\"\"\"Small decoder-only Transformer trained from random initialization; no pretrained weights.\"\"\"\nfrom dataclasses import asdict, dataclass\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass\nclass ModelConfig:\n    vocab_size: int = 16000\n    block_size: int = 1024\n    n_layer: int = 6\n    n_head: int = 8\n    n_embd: int = 256\n    dropout: float = 0.1\n\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        if config.n_embd % config.n_head:\n            raise ValueError(\"Embedding size must be divisible by head count.\")\n        self.n_head = config.n_head\n        self.head_dim = config.n_embd // config.n_head\n        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)\n        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)\n        self.dropout = config.dropout\n\n    def forward(self, x):\n        batch, length, channels = x.shape\n        q, k, v = self.qkv(x).split(channels, dim=2)\n        q = q.view(batch, length, self.n_head, self.head_dim).transpose(1, 2)\n        k = k.view(batch, length, self.n_head, self.head_dim).transpose(1, 2)\n        v = v.view(batch, length, self.n_head, self.head_dim).transpose(1, 2)\n        value = F.scaled_dot_product_attention(\n            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True\n        )\n        value = value.transpose(1, 2).contiguous().view(batch, length, channels)\n        return self.proj(value)\n\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(config.n_embd)\n        self.attention = CausalSelfAttention(config)\n        self.norm2 = nn.LayerNorm(config.n_embd)\n        self.feed_forward = nn.Sequential(\n            nn.Linear(config.n_embd, 4 * config.n_embd),\n            nn.GELU(),\n            nn.Linear(4 * config.n_embd, config.n_embd),\n            nn.Dropout(config.dropout),\n        )\n\n    def forward(self, x):\n        x = x + self.attention(self.norm1(x))\n        return x + self.feed_forward(self.norm2(x))\n\n\nclass Expert802GPT(nn.Module):\n    def __init__(self, config=None):\n        super().__init__()\n        self.config = config or ModelConfig()\n        self.token_embedding = nn.Embedding(self.config.vocab_size, self.config.n_embd)\n        self.position_embedding = nn.Embedding(self.config.block_size, self.config.n_embd)\n        self.dropout = nn.Dropout(self.config.dropout)\n        self.blocks = nn.Sequential(*[TransformerBlock(self.config) for _ in range(self.config.n_layer)])\n        self.final_norm = nn.LayerNorm(self.config.n_embd)\n        self.language_head = nn.Linear(self.config.n_embd, self.config.vocab_size, bias=False)\n        self.language_head.weight = self.token_embedding.weight\n        self.apply(self._initialize)\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module, nn.Linear):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n\n    def forward(self, tokens, targets=None, loss_mask=None):\n        batch, length = tokens.shape\n        if length > self.config.block_size:\n            raise ValueError(\"Input exceeds configured context length.\")\n        positions = torch.arange(length, device=tokens.device)\n        hidden = self.dropout(self.token_embedding(tokens) + self.position_embedding(positions))\n        hidden = self.final_norm(self.blocks(hidden))\n        logits = self.language_head(hidden)\n        loss = None\n        if targets is not None:\n            token_losses = F.cross_entropy(\n                logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction=\"none\"\n            ).view(batch, length)\n            if loss_mask is not None:\n                weights = loss_mask.to(dtype=token_losses.dtype)\n                denominator = weights.sum()\n                if denominator.item() <= 0:\n                    raise ValueError(\"A supervised batch must contain at least one assistant target token.\")\n                loss = (token_losses * weights).sum() / denominator\n            else:\n                loss = token_losses.mean()\n        return logits, loss\n\n    @torch.no_grad()\n    def generate(self, tokens, max_new_tokens=300, temperature=0.8, top_k=40, stop_token_id=None):\n        self.eval()\n        for _ in range(max_new_tokens):\n            context = tokens[:, -self.config.block_size:]\n            logits, _ = self(context)\n            logits = logits[:, -1, :] / max(temperature, 1e-4)\n            if top_k:\n                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))\n                logits[logits < values[:, [-1]]] = float(\"-inf\")\n            probabilities = F.softmax(logits, dim=-1)\n            next_token = torch.multinomial(probabilities, num_samples=1)\n            tokens = torch.cat((tokens, next_token), dim=1)\n            if stop_token_id is not None and int(next_token[0, 0]) == int(stop_token_id):\n                break\n        return tokens\n\n    def parameter_count(self):\n        return sum(parameter.numel() for parameter in self.parameters())\n\n    def config_dict(self):\n        return asdict(self.config)\n",
  "tokenizer.py": "\"\"\"Locally trained byte-level BPE tokenizer. No downloaded vocabulary or hosted service.\"\"\"\nfrom tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers\n\nSPECIAL_TOKENS = [\"<|endoftext|>\", \"<|system|>\", \"<|user|>\", \"<|assistant|>\", \"<|tool|>\"]\n\ndef train_tokenizer(files, output_path, vocab_size=16000, min_frequency=2):\n    tokenizer = Tokenizer(models.BPE(unk_token=\"<|unk|>\"))\n    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\n    tokenizer.decoder = decoders.ByteLevel()\n    trainer = trainers.BpeTrainer(\n        vocab_size=vocab_size,\n        min_frequency=min_frequency,\n        special_tokens=[\"<|unk|>\"] + SPECIAL_TOKENS,\n        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),\n        show_progress=True,\n    )\n    tokenizer.train(files=[str(path) for path in files], trainer=trainer)\n    tokenizer.save(str(output_path))\n    return tokenizer\n\ndef load_tokenizer(path):\n    return Tokenizer.from_file(str(path))\n\ndef encode_text(tokenizer, text):\n    return tokenizer.encode(text).ids\n\ndef decode_tokens(tokenizer, ids):\n    return tokenizer.decode([int(value) for value in ids], skip_special_tokens=False)\n",
  "test_model.py": "\"\"\"Tiny forward/backward/generation smoke test for the native Transformer.\"\"\"\nimport torch\nfrom model import Expert802GPT, ModelConfig\n\ndef main():\n    torch.manual_seed(802)\n    config=ModelConfig(vocab_size=512,block_size=64,n_layer=2,n_head=4,n_embd=64,dropout=0.0)\n    model=Expert802GPT(config)\n    tokens=torch.randint(0,config.vocab_size,(2,32))\n    logits,loss=model(tokens,tokens)\n    assert logits.shape==(2,32,config.vocab_size)\n    assert loss is not None and torch.isfinite(loss)\n    loss.backward()\n    assert any(parameter.grad is not None for parameter in model.parameters())\n    generated=model.generate(tokens[:1,:8],max_new_tokens=3)\n    assert generated.shape==(1,11)\n    print({\"parameters\":model.parameter_count(),\"loss\":float(loss.item()),\"generation_shape\":list(generated.shape)})\n\nif __name__==\"__main__\":\n    main()\n",
  "evaluate.py": "\"\"\"Evaluate a trained Expert 802 checkpoint on held-out tokens and fixed prompts.\"\"\"\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom generate import load_model\nfrom tokenizer import load_tokenizer\n\n\n@torch.no_grad()\ndef evaluate(checkpoint_path, data_dir, tokenizer_path, batch_size=4, batches=50, prompts=None):\n    device = \"cuda\" if torch.cuda.is_available() else (\"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    model, payload = load_model(checkpoint_path, device)\n    metadata = json.loads((Path(data_dir) / \"metadata.json\").read_text(encoding=\"utf-8\"))\n    validation = np.memmap(\n        Path(data_dir) / \"val.bin\",\n        dtype=np.dtype(metadata[\"token_dtype\"]),\n        mode=\"r\",\n    )\n    if len(validation) <= model.config.block_size + 1:\n        raise ValueError(\"Validation split is shorter than the model context.\")\n    losses = []\n    model.eval()\n    for _ in range(max(1, batches)):\n        starts = torch.randint(0, len(validation) - model.config.block_size - 1, (batch_size,))\n        x = torch.stack([\n            torch.from_numpy(np.array(validation[i:i + model.config.block_size], dtype=np.int64))\n            for i in starts\n        ]).to(device)\n        y = torch.stack([\n            torch.from_numpy(np.array(validation[i + 1:i + model.config.block_size + 1], dtype=np.int64))\n            for i in starts\n        ]).to(device)\n        _, loss = model(x, y)\n        losses.append(float(loss.item()))\n\n    tokenizer = load_tokenizer(tokenizer_path)\n    sample_prompts = prompts or [\n        \"<|user|> Hello. Please introduce yourself.\\n<|assistant|>\",\n        \"<|user|> Explain what a Python function is.\\n<|assistant|>\",\n        \"<|user|> Write a JavaScript function that adds two numbers.\\n<|assistant|>\",\n        \"<|user|> I made a mistake in my code. How should I debug it?\\n<|assistant|>\",\n    ]\n    samples = []\n    for prompt in sample_prompts:\n        ids = tokenizer.encode(prompt).ids[-model.config.block_size:]\n        if not ids:\n            continue\n        context = torch.tensor([ids], dtype=torch.long, device=device)\n        generated = model.generate(\n            context,\n            max_new_tokens=96,\n            temperature=0.7,\n            top_k=40,\n            stop_token_id=tokenizer.token_to_id(\"<|endoftext|>\"),\n        )\n        completion = tokenizer.decode(generated[0].tolist()[len(ids):], skip_special_tokens=True)\n        samples.append({\"prompt\": prompt, \"completion\": completion})\n\n    mean_loss = sum(losses) / len(losses)\n    return {\n        \"checkpoint\": str(checkpoint_path),\n        \"step\": int(payload.get(\"step\", 0)),\n        \"device\": device,\n        \"parameters\": model.parameter_count(),\n        \"validation_loss\": mean_loss,\n        \"validation_perplexity\": math.exp(min(mean_loss, 20)),\n        \"validation_batches\": len(losses),\n        \"samples\": samples,\n        \"warning\": \"Loss and perplexity measure next-token prediction, not conversational helpfulness or coding correctness. Review generated samples and run task-specific tests before deployment.\",\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--checkpoint\", default=\"training/checkpoints/best.pt\")\n    parser.add_argument(\"--data\", default=\"training/data\")\n    parser.add_argument(\"--tokenizer\", default=\"training/checkpoints/tokenizer.json\")\n    parser.add_argument(\"--batches\", type=int, default=50)\n    parser.add_argument(\"--batch-size\", type=int, default=4)\n    parser.add_argument(\"--output\", default=\"\")\n    args = parser.parse_args()\n    result = evaluate(args.checkpoint, args.data, args.tokenizer, args.batch_size, args.batches)\n    rendered = json.dumps(result, ensure_ascii=False, indent=2)\n    if args.output:\n        Path(args.output).parent.mkdir(parents=True, exist_ok=True)\n        Path(args.output).write_text(rendered + \"\\n\", encoding=\"utf-8\")\n    print(rendered)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "train.py": "\"\"\"Train Expert802GPT with optional assistant-only supervised loss masks.\"\"\"\nimport argparse\nimport json\nimport math\nimport random\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom model import Expert802GPT, ModelConfig\n\n\ndef load_tokens(path, dtype):\n    return np.memmap(Path(path), dtype=np.dtype(dtype), mode=\"r\")\n\n\ndef get_batch(data, batch_size, block_size, device, mask_data=None):\n    if len(data) <= block_size + 1:\n        raise ValueError(\"Token file is shorter than the configured context.\")\n    starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))\n    x = torch.stack([torch.from_numpy(np.array(data[i:i + block_size], dtype=np.int64)) for i in starts])\n    y = torch.stack([torch.from_numpy(np.array(data[i + 1:i + block_size + 1], dtype=np.int64)) for i in starts])\n    x, y = x.to(device), y.to(device)\n    if mask_data is None:\n        return x, y, None\n    weights = torch.stack([\n        torch.from_numpy(np.array(mask_data[i + 1:i + block_size + 1], dtype=np.float32))\n        for i in starts\n    ]).to(device)\n    if not torch.any(weights):\n        for _ in range(31):\n            starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))\n            x = torch.stack([torch.from_numpy(np.array(data[i:i + block_size], dtype=np.int64)) for i in starts])\n            y = torch.stack([torch.from_numpy(np.array(data[i + 1:i + block_size + 1], dtype=np.int64)) for i in starts])\n            weights = torch.stack([\n                torch.from_numpy(np.array(mask_data[i + 1:i + block_size + 1], dtype=np.float32))\n                for i in starts\n            ])\n            if torch.any(weights):\n                return x.to(device), y.to(device), weights.to(device)\n        raise ValueError(\"Could not sample assistant targets. Increase the number and density of reviewed assistant responses.\")\n    return x, y, weights\n\n\n@torch.no_grad()\ndef estimate_loss(model, train_data, val_data, batch_size, block_size, device,\n                  train_mask=None, val_mask=None, batches=20):\n    model.eval()\n    results = {}\n    for name, data, mask in ((\"train\", train_data, train_mask), (\"validation\", val_data, val_mask)):\n        losses = []\n        for _ in range(batches):\n            x, y, weights = get_batch(data, batch_size, block_size, device, mask)\n            _, loss = model(x, y, weights)\n            losses.append(float(loss.item()))\n        results[name] = sum(losses) / len(losses)\n    model.train()\n    return results\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--data\", default=\"training/data\")\n    parser.add_argument(\"--output\", default=\"training/checkpoints\")\n    parser.add_argument(\"--steps\", type=int, default=10000)\n    parser.add_argument(\"--batch-size\", type=int, default=16)\n    parser.add_argument(\"--block-size\", type=int, default=1024)\n    parser.add_argument(\"--layers\", type=int, default=6)\n    parser.add_argument(\"--heads\", type=int, default=8)\n    parser.add_argument(\"--width\", type=int, default=256)\n    parser.add_argument(\"--learning-rate\", type=float, default=3e-4)\n    parser.add_argument(\"--eval-interval\", type=int, default=200)\n    parser.add_argument(\"--resume\", default=\"\")\n    args = parser.parse_args()\n    torch.manual_seed(802)\n    random.seed(802)\n    device = \"cuda\" if torch.cuda.is_available() else (\"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    data_dir = Path(args.data)\n    metadata = json.loads((data_dir / \"metadata.json\").read_text(encoding=\"utf-8\"))\n    checkpoint = torch.load(args.resume, map_location=device, weights_only=False) if args.resume else None\n    if checkpoint:\n        config = ModelConfig(**checkpoint[\"config\"])\n        if config.vocab_size != int(metadata[\"vocab_size\"]):\n            raise ValueError(\"The fine-tuning tokenizer vocabulary must match the base checkpoint.\")\n    else:\n        config = ModelConfig(vocab_size=int(metadata[\"vocab_size\"]), block_size=args.block_size,\n                            n_layer=args.layers, n_head=args.heads, n_embd=args.width)\n    model = Expert802GPT(config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)\n    start_step = 0\n    if checkpoint:\n        model.load_state_dict(checkpoint[\"model\"])\n        if \"optimizer\" in checkpoint:\n            optimizer.load_state_dict(checkpoint[\"optimizer\"])\n        start_step = int(checkpoint.get(\"step\", 0))\n    train_data = load_tokens(data_dir / \"train.bin\", metadata[\"token_dtype\"])\n    val_data = load_tokens(data_dir / \"val.bin\", metadata[\"token_dtype\"])\n    train_mask_path, val_mask_path = data_dir / \"train_mask.bin\", data_dir / \"val_mask.bin\"\n    train_mask = np.memmap(train_mask_path, dtype=np.uint8, mode=\"r\") if train_mask_path.exists() else None\n    val_mask = np.memmap(val_mask_path, dtype=np.uint8, mode=\"r\") if val_mask_path.exists() else None\n    if (train_mask is None) != (val_mask is None):\n        raise ValueError(\"Training and validation loss masks must either both exist or both be absent.\")\n    if train_mask is not None and (len(train_mask) != len(train_data) or len(val_mask) != len(val_data)):\n        raise ValueError(\"Loss masks must have exactly one entry per token.\")\n    if len(train_data) <= args.block_size + 1 or len(val_data) <= args.block_size + 1:\n        raise ValueError(\"Both train and validation splits must exceed block_size + 1 tokens.\")\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(data_dir / \"tokenizer.json\", output / \"tokenizer.json\")\n    print(json.dumps({\"device\": device, \"parameters\": model.parameter_count(), \"config\": model.config_dict(),\n                      \"train_tokens\": len(train_data), \"validation_tokens\": len(val_data),\n                      \"objective\": metadata.get(\"objective\", \"all_token_next_token_prediction\")}))\n    best_validation = float(\"inf\")\n    best_checkpoint_path = output / \"best.pt\"\n    if best_checkpoint_path.exists():\n        previous_best = torch.load(best_checkpoint_path, map_location=\"cpu\", weights_only=False)\n        best_validation = float(previous_best.get(\"metrics\", {}).get(\"validation\", float(\"inf\")))\n    for step in range(start_step, args.steps):\n        progress = step / max(1, args.steps)\n        learning_rate = args.learning_rate * 0.1 + 0.9 * args.learning_rate * 0.5 * (1 + math.cos(math.pi * progress))\n        for group in optimizer.param_groups:\n            group[\"lr\"] = learning_rate\n        x, y, weights = get_batch(train_data, args.batch_size, args.block_size, device, train_mask)\n        optimizer.zero_grad(set_to_none=True)\n        _, loss = model(x, y, weights)\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        optimizer.step()\n        if step % args.eval_interval == 0 or step == args.steps - 1:\n            metrics = estimate_loss(model, train_data, val_data, args.batch_size, args.block_size, device,\n                                     train_mask, val_mask)\n            print(json.dumps({\"step\": step, \"loss\": float(loss.item()), **metrics}))\n            payload = {\"model\": model.state_dict(), \"optimizer\": optimizer.state_dict(), \"step\": step + 1,\n                       \"config\": model.config_dict(), \"metrics\": metrics,\n                       \"objective\": metadata.get(\"objective\", \"all_token_next_token_prediction\"),\n                       \"tokenizer\": \"byte-level-bpe-v1\"}\n            torch.save(payload, output / \"latest.pt\")\n            if metrics[\"validation\"] < best_validation:\n                best_validation = metrics[\"validation\"]\n                torch.save(payload, output / \"best.pt\")\n    print(\"Training complete. Checkpoints are stored outside the web application.\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "prepare_data.py": "\"\"\"Prepare a locally tokenized corpus with source-level train/validation isolation.\"\"\"\nimport argparse\nimport json\nimport random\nimport tempfile\nfrom pathlib import Path\nimport numpy as np\nfrom tokenizer import train_tokenizer, load_tokenizer, encode_text\n\nTEXT_SUFFIXES = {\".txt\", \".md\", \".py\", \".js\", \".jsx\", \".ts\", \".tsx\", \".html\", \".css\", \".sql\", \".json\", \".jsonl\"}\n\ndef read_source(path):\n    if path.suffix.lower() == \".jsonl\":\n        conversations = []\n        for line in path.read_text(encoding=\"utf-8\", errors=\"ignore\").splitlines():\n            try:\n                item = json.loads(line)\n            except json.JSONDecodeError:\n                continue\n            messages = item.get(\"messages\", []) if isinstance(item, dict) else []\n            turns = []\n            for message in messages if isinstance(messages, list) else []:\n                if isinstance(message, dict) and message.get(\"role\") in {\"system\", \"user\", \"assistant\"}:\n                    body = message.get(\"content\", \"\")\n                    if isinstance(body, str) and body.strip():\n                        turns.append(\"<|\" + message[\"role\"] + \"|> \" + body.strip())\n            if turns:\n                conversations.append(\"\\n\".join(turns))\n        return \"\\n\\n\".join(conversations)\n    if path.suffix.lower() in TEXT_SUFFIXES:\n        return path.read_text(encoding=\"utf-8\", errors=\"ignore\")\n    return \"\"\n\ndef split_documents(text):\n    pieces = [part.strip() for part in text.split(\"\\n\\n\") if len(part.strip()) > 80]\n    if len(pieces) > 1:\n        return pieces\n    lines = [line.strip() for line in text.splitlines() if len(line.strip()) > 40]\n    return lines if len(lines) > 1 else ([text.strip()] if text.strip() else [])\n\ndef prepare(sources, output_dir, validation_fraction=0.02, vocab_size=16000, seed=802, tokenizer_path=None):\n    source_groups = []\n    for source in sources:\n        path = Path(source)\n        candidates = [path] if path.is_file() else (\n            sorted((item for item in path.rglob(\"*\") if item.is_file() and not item.is_symlink()\n                    and item.suffix.lower() in TEXT_SUFFIXES), key=lambda item: item.as_posix()) if path.is_dir() else [])\n        for candidate in candidates:\n            if candidate.stat().st_size > 20_000_000:\n                continue\n            docs = split_documents(read_source(candidate))\n            if docs:\n                source_groups.append((candidate.as_posix(), docs))\n    if len(source_groups) < 2:\n        raise ValueError(\"At least two independent source files are required to prevent train/validation leakage.\")\n    random.Random(seed).shuffle(source_groups)\n    validation_count = max(1, int(len(source_groups) * validation_fraction))\n    validation_groups = source_groups[:validation_count]\n    training_groups = source_groups[validation_count:]\n    training_docs = [doc for _, docs in training_groups for doc in docs]\n    validation_docs = [doc for _, docs in validation_groups for doc in docs]\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    if tokenizer_path:\n        tokenizer = load_tokenizer(tokenizer_path)\n        import shutil\n        shutil.copy2(tokenizer_path, output / \"tokenizer.json\")\n    else:\n        with tempfile.NamedTemporaryFile(\"w\", encoding=\"utf-8\", suffix=\".txt\", delete=False) as handle:\n            corpus_path = Path(handle.name)\n            handle.write(\"\\n\\n\".join(training_docs))\n        try:\n            tokenizer = train_tokenizer([corpus_path], output / \"tokenizer.json\", vocab_size=vocab_size)\n        finally:\n            corpus_path.unlink(missing_ok=True)\n    separator = tokenizer.token_to_id(\"<|endoftext|>\")\n    def tokenize(documents):\n        ids = []\n        for document in documents:\n            ids.extend(encode_text(tokenizer, document))\n            ids.append(separator)\n        return ids\n    train_ids, validation_ids = tokenize(training_docs), tokenize(validation_docs)\n    if len(train_ids) <= 1025 or len(validation_ids) <= 1025:\n        raise ValueError(\"Both source-isolated splits must exceed 1025 tokens for the default 1,024-token context.\")\n    dtype = np.uint16 if tokenizer.get_vocab_size() <= 65535 else np.uint32\n    np.asarray(train_ids, dtype=dtype).tofile(output / \"train.bin\")\n    np.asarray(validation_ids, dtype=dtype).tofile(output / \"val.bin\")\n    metadata = {\"tokenizer\": \"byte-level-bpe-v1\", \"vocab_size\": tokenizer.get_vocab_size(),\n                \"token_dtype\": str(np.dtype(dtype)), \"train_tokens\": len(train_ids),\n                \"validation_tokens\": len(validation_ids), \"training_documents\": len(training_docs),\n                \"validation_documents\": len(validation_docs), \"training_source_files\": len(training_groups),\n                \"validation_source_files\": len(validation_groups), \"split_unit\": \"source_file\"}\n    (output / \"metadata.json\").write_text(json.dumps(metadata, indent=2), encoding=\"utf-8\")\n    return metadata\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"sources\", nargs=\"+\", help=\"Explicit reviewed text/code corpus folders or files.\")\n    parser.add_argument(\"--output\", default=\"training/data\")\n    parser.add_argument(\"--vocab-size\", type=int, default=16000)\n    parser.add_argument(\"--tokenizer\", default=\"\", help=\"Reuse an existing tokenizer to keep resumed training token IDs compatible.\")\n    args = parser.parse_args()\n    print(json.dumps(prepare(args.sources, args.output, vocab_size=args.vocab_size,\n                             tokenizer_path=args.tokenizer or None), indent=2))\n\nif __name__ == \"__main__\":\n    main()\n",
  "audit_corpus.py": "\"\"\"Audit a text/code corpus before training; require explicit provenance and license records.\"\"\"\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nTEXT_SUFFIXES = {\".txt\", \".md\", \".py\", \".js\", \".jsx\", \".ts\", \".tsx\", \".html\", \".css\", \".sql\", \".json\", \".jsonl\"}\nSECRET_PATTERNS = [\n    re.compile(r\"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----\"),\n    re.compile(r\"\\bAKIA[0-9A-Z]{16}\\b\"),\n    re.compile(r\"\\bgh[pousr]_[A-Za-z0-9]{30,}\\b\"),\n    re.compile(r\"(?i)(?:api[_-]?key|secret|password|access[_-]?token)\\s*[:=]\\s*['\\\"][^'\\\"]{12,}\"),\n]\n\ndef audit(corpus_dir, manifest_path, output_path, max_file_mb=20):\n    root = Path(corpus_dir).resolve()\n    manifest = {}\n    with Path(manifest_path).open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as handle:\n        for row in csv.DictReader(handle):\n            rel = (row.get(\"path\") or \"\").strip()\n            if rel:\n                manifest[rel.replace(\"\\\\\", \"/\")] = row\n    report = {\"corpus\": str(root), \"files_scanned\": 0, \"accepted_files\": 0,\n              \"rejected\": [], \"warnings\": [], \"accepted\": [], \"duplicate_groups\": []}\n    hashes = {}\n    for path in sorted(root.rglob(\"*\")):\n        if not path.is_file():\n            continue\n        if path.is_symlink():\n            report[\"rejected\"].append({\"path\": str(path.relative_to(root)), \"reason\": \"symbolic links are not accepted\"})\n            continue\n        rel = path.relative_to(root).as_posix()\n        if path.suffix.lower() not in TEXT_SUFFIXES:\n            report[\"rejected\"].append({\"path\": rel, \"reason\": \"unsupported file type\"})\n            continue\n        if path.stat().st_size == 0 or path.stat().st_size > max_file_mb * 1024 * 1024:\n            report[\"rejected\"].append({\"path\": rel, \"reason\": \"empty or exceeds size limit\"})\n            continue\n        try:\n            raw = path.read_bytes()\n            content = raw.decode(\"utf-8\")\n        except (OSError, UnicodeDecodeError):\n            report[\"rejected\"].append({\"path\": rel, \"reason\": \"not valid UTF-8 text\"})\n            continue\n        report[\"files_scanned\"] += 1\n        row = manifest.get(rel)\n        if not row or not (row.get(\"license\") or \"\").strip() or not (row.get(\"permission_basis\") or \"\").strip():\n            report[\"rejected\"].append({\"path\": rel, \"reason\": \"missing license and permission_basis in corpus manifest\"})\n            continue\n        if any(pattern.search(content) for pattern in SECRET_PATTERNS):\n            report[\"rejected\"].append({\"path\": rel, \"reason\": \"possible credential or private key detected\"})\n            continue\n        digest = hashlib.sha256(raw).hexdigest()\n        if digest in hashes:\n            report[\"duplicate_groups\"].append({\"sha256\": digest, \"paths\": [hashes[digest], rel]})\n            report[\"rejected\"].append({\"path\": rel, \"reason\": \"duplicate content\"})\n            continue\n        hashes[digest] = rel\n        item = {\"path\": rel, \"sha256\": digest, \"bytes\": len(raw),\n                \"license\": row[\"license\"].strip(),\n                \"source_url\": (row.get(\"source_url\") or \"\").strip(),\n                \"permission_basis\": row[\"permission_basis\"].strip()}\n        report[\"accepted\"].append(item)\n        report[\"accepted_files\"] += 1\n        if not item[\"source_url\"]:\n            report[\"warnings\"].append({\"path\": rel, \"reason\": \"source_url is empty; retain independent provenance records\"})\n    report[\"rejected_files\"] = len(report[\"rejected\"])\n    Path(output_path).parent.mkdir(parents=True, exist_ok=True)\n    Path(output_path).write_text(json.dumps(report, ensure_ascii=False, indent=2) + \"\\n\", encoding=\"utf-8\")\n    print(json.dumps({k: report[k] for k in (\"files_scanned\", \"accepted_files\", \"rejected_files\", \"warnings\")}, indent=2))\n    if report[\"accepted_files\"] == 0 or report[\"rejected_files\"]:\n        raise SystemExit(\"Corpus audit failed: fix rejected files or provide a fully reviewed corpus before training.\")\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"corpus\")\n    parser.add_argument(\"--manifest\", required=True, help=\"CSV with path,license,source_url,permission_basis,notes columns\")\n    parser.add_argument(\"--output\", default=\"training/corpus_audit.json\")\n    parser.add_argument(\"--max-file-mb\", type=int, default=20)\n    args = parser.parse_args()\n    audit(args.corpus, args.manifest, args.output, args.max_file_mb)\n\nif __name__ == \"__main__\":\n    main()\n",
  "generate.py": "\"\"\"Generate text from a trained Expert802GPT checkpoint.\"\"\"\nimport argparse\nimport torch\nfrom model import Expert802GPT,ModelConfig\nfrom tokenizer import load_tokenizer\n\ndef load_model(checkpoint_path,device):\n    payload=torch.load(checkpoint_path,map_location=device,weights_only=False)\n    config=ModelConfig(**payload[\"config\"])\n    model=Expert802GPT(config).to(device)\n    model.load_state_dict(payload[\"model\"])\n    model.eval()\n    return model,payload\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\"--checkpoint\",default=\"training/checkpoints/best.pt\")\n    parser.add_argument(\"--tokenizer\",default=\"training/checkpoints/tokenizer.json\")\n    parser.add_argument(\"--prompt\",default=\"<|user|> Hello.\\n<|assistant|>\")\n    parser.add_argument(\"--tokens\",type=int,default=200)\n    parser.add_argument(\"--temperature\",type=float,default=0.8)\n    args=parser.parse_args()\n    device=\"cuda\" if torch.cuda.is_available() else (\"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    model,payload=load_model(args.checkpoint,device)\n    tokenizer=load_tokenizer(args.tokenizer)\n    prompt_ids=tokenizer.encode(args.prompt).ids[-model.config.block_size:]\n    context=torch.tensor([prompt_ids],dtype=torch.long,device=device)\n    result=model.generate(context,max_new_tokens=max(1,min(args.tokens,8192)),temperature=args.temperature,stop_token_id=tokenizer.token_to_id(\"<|endoftext|>\"))\n    print(tokenizer.decode(result[0].tolist()[len(prompt_ids):],skip_special_tokens=True))\n\nif __name__==\"__main__\":\n    main()\n"
}
training_dir = WORKSPACE / "training"
training_dir.mkdir(parents=True, exist_ok=True)
for filename, source in SOURCE_FILES.items():
    (training_dir / filename).write_text(source, encoding="utf-8")
print("Installed embedded Expert 802 training files:", ", ".join(sorted(SOURCE_FILES)))


In [ ]:
import torch, numpy
try:
    import tokenizers
except ImportError as exc:
    raise RuntimeError("The Kaggle image is missing the 'tokenizers' package. Enable Internet in Notebook Options and restart, or use a Kaggle image that already includes tokenizers.") from exc
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Open Notebook Settings, set Accelerator to GPU, save, then restart and run all cells.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "NumPy:", numpy.__version__, "tokenizers:", tokenizers.__version__)
subprocess.run([sys.executable, "training/test_model.py"], check=True)


In [ ]:
from pathlib import Path
import shutil, json, subprocess, sys, zipfile

if not CORPUS_DIR.is_dir() or not MANIFEST.is_file():
    raise FileNotFoundError(f"Expected {CORPUS_DIR} and {MANIFEST}. Check CORPUS_DATASET and your attached Kaggle Dataset layout.")

checkpoint_dir = WORKSPACE / "training" / "checkpoints"
data_dir = WORKSPACE / "training" / "data"
audit_file = WORKSPACE / "training" / "corpus_audit.json"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

if START_FRESH:
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    shutil.rmtree(data_dir, ignore_errors=True)

# Restore the previous private session bundle, if one was attached.
previous_zip = list(PREVIOUS_DIR.rglob("expert802-session.zip")) if PREVIOUS_DIR.exists() else []
if previous_zip and not START_FRESH:
    restore_dir = Path("/kaggle/working/expert802_restore")
    shutil.rmtree(restore_dir, ignore_errors=True)
    restore_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(previous_zip[0]) as archive:
        archive.extractall(restore_dir)
    restored = restore_dir / "checkpoints"
    if restored.is_dir():
        shutil.copytree(restored, checkpoint_dir, dirs_exist_ok=True)
    previous_audit = restore_dir / "corpus_audit.json"
else:
    previous_audit = None

# Audit the current corpus every session. Provenance is recorded; this is a screening gate, not legal advice.
subprocess.run([sys.executable, "training/audit_corpus.py", str(CORPUS_DIR), "--manifest", str(MANIFEST), "--output", str(audit_file)], check=True)
if previous_audit and previous_audit.exists():
    old = json.loads(previous_audit.read_text(encoding="utf-8"))
    new = json.loads(audit_file.read_text(encoding="utf-8"))
    old_signature = [(x["path"], x["sha256"], x["license"]) for x in old.get("accepted", [])]
    new_signature = [(x["path"], x["sha256"], x["license"]) for x in new.get("accepted", [])]
    if old_signature != new_signature:
        raise RuntimeError("The corpus differs from the previous session. Set START_FRESH=True to begin a separate run, or attach the exact same corpus.")

# Rebuild token files from the same corpus. Reuse the saved tokenizer on resumed runs so token IDs never drift.
saved_tokenizer = checkpoint_dir / "tokenizer.json"
prepare_cmd = [sys.executable, "training/prepare_data.py", str(CORPUS_DIR), "--output", str(data_dir), "--vocab-size", str(VOCAB_SIZE)]
if saved_tokenizer.exists():
    prepare_cmd += ["--tokenizer", str(saved_tokenizer)]
subprocess.run(prepare_cmd, check=True)
print("Prepared training data:", data_dir)


In [ ]:
train_cmd = [
    sys.executable, "training/train.py",
    "--data", str(data_dir), "--output", str(checkpoint_dir),
    "--steps", str(TOTAL_STEPS), "--batch-size", str(BATCH_SIZE),
    "--block-size", str(CONTEXT_LENGTH), "--layers", "6", "--heads", "8", "--width", "256",
    "--eval-interval", "100",
]
latest = checkpoint_dir / "latest.pt"
if latest.exists():
    train_cmd += ["--resume", str(latest)]
    print("Resuming from the saved checkpoint; target total steps:", TOTAL_STEPS)
else:
    print("Starting from random initialization; this is not a pretrained model.")
subprocess.run(train_cmd, check=True)

subprocess.run([
    sys.executable, "training/evaluate.py",
    "--checkpoint", str(checkpoint_dir / "best.pt"),
    "--data", str(data_dir),
    "--tokenizer", str(checkpoint_dir / "tokenizer.json"),
    "--output", str(checkpoint_dir / "evaluation.json"),
], check=True)


In [ ]:
# Export only model checkpoints, tokenizer, evaluation and corpus provenance report.
# The original corpus and tokenized training data are deliberately NOT bundled.
export_dir = Path("/kaggle/working/expert802_export")
shutil.rmtree(export_dir, ignore_errors=True)
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(checkpoint_dir, export_dir / "checkpoints")
shutil.copy2(audit_file, export_dir / "corpus_audit.json")
zip_path = Path("/kaggle/working/expert802-session.zip")
zip_path.unlink(missing_ok=True)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in export_dir.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(export_dir))
print("Session bundle ready:", zip_path)
print("Download expert802-session.zip from the Kaggle notebook Output panel. Keep it private and attach it to your next session to resume.")


## Important expectations
- This is genuine training from random initialization, but the current 6-layer, 256-wide model is a small experimental model, not ChatGPT-level.
- A low training loss does not prove factual accuracy, reasoning ability, or working code. Review `checkpoints/evaluation.json` and test generated samples.
- Kaggle sessions can end or GPU access can be unavailable. Save the ZIP output after every session and attach it next time.
- Do not publish checkpoints or corpus files containing private or restricted material. Keep your Kaggle datasets private when required by their licenses.
